# U-Sleep внутри ансамбля:160–165

Один Run All: сначала пять быстрых абляций, таблица и отдельный ZIP; затем автоматически Gate4 + TCN4.
T4×2, Internet On, прежний GITHUB_TOKEN. Подключить исходный Sleep-EDF и новый sleep-edf-usleep-integration-inputs.zip.
Старые113/156–159 не изменяются. Normalization/CV только внутри train; validation используется лишь для сравнения.


In [ ]:
import json
import os
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

REPO_URL = "https://github.com/revlze/sleep-edf-classification.git"
CODE_REF = "feat/usleep-rbd-pd"
PROJECT_ROOT = Path("/tmp/sleep-edf-usleep-integration-code-v1")
OUTPUT_ROOT = Path("/kaggle/working/usleep-integration-results")
INPUT_ROOT = None  # Новый sleep-edf-usleep-integration-inputs.zip.
RESUME_PHASE1 = None  # Необязательно: ZIP результатов Phase1 той же версии кода.
RESUME_PHASE2 = None  # Необязательно: ZIP Phase2 с checkpoints и train U-Sleep cache.
RUN_PHASE2 = True  # После Phase1 ZIP автоматически перейти к обучению.
ALLOW_TRAINING = True  # Разрешено только обучение двух новых heads.
DEVICE = "cuda:0"
THREADS = max(1, os.cpu_count() or 2)


def checkout_code():
    if PROJECT_ROOT.exists():
        current_ref = subprocess.check_output(
            ["git", "config", "sleepedf.bootstrapRef"], cwd=PROJECT_ROOT, text=True
        ).strip()
        if current_ref != CODE_REF:
            raise RuntimeError(
                "Для другого CODE_REF задайте новый PROJECT_ROOT в /tmp."
            )
        print(
            "Используется существующий checkout; для обновления ветки задайте новый PROJECT_ROOT."
        )
        return
    from kaggle_secrets import UserSecretsClient

    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    with tempfile.TemporaryDirectory(prefix="sleep-git-") as temporary:
        askpass = Path(temporary) / "askpass.py"
        askpass.write_text(
            "#!/usr/bin/env python3\nimport os, sys\n"
            'print("x-access-token" if "Username" in sys.argv[1] else os.environ["SLEEP_GITHUB_TOKEN"])\n'
        )
        askpass.chmod(0o700)
        env = {
            **os.environ,
            "GIT_ASKPASS": str(askpass),
            "GIT_TERMINAL_PROMPT": "0",
            "SLEEP_GITHUB_TOKEN": token,
        }
        subprocess.run(
            ["git", "init", "--quiet", "--initial-branch=bootstrap", str(PROJECT_ROOT)],
            check=True,
        )
        subprocess.run(
            ["git", "remote", "add", "origin", REPO_URL], cwd=PROJECT_ROOT, check=True
        )
        subprocess.run(
            [
                "git",
                "-c",
                "credential.helper=",
                "fetch",
                "--quiet",
                "--depth",
                "1",
                "origin",
                CODE_REF,
            ],
            cwd=PROJECT_ROOT,
            env=env,
            check=True,
        )
        subprocess.run(
            ["git", "checkout", "--quiet", "--detach", "FETCH_HEAD"],
            cwd=PROJECT_ROOT,
            check=True,
        )
        subprocess.run(
            ["git", "config", "sleepedf.bootstrapRef", CODE_REF],
            cwd=PROJECT_ROOT,
            check=True,
        )
    del token


checkout_code()
commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True
).strip()
print("Код:", commit)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.notebook import reset_commands, run_command  # noqa: E402

reset_commands()

# Устанавливаем только инструмент uv; зависимости проекта устанавливает uv.lock.

UV_VERSION = "0.12.15"
if shutil.which("uv") is None:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", f"uv=={UV_VERSION}"], check=True
    )
UV = shutil.which("uv")
ENV = {
    "UV_PROJECT_ENVIRONMENT": "/tmp/sleep-edf-usleep-integration-venv",
    "UV_CACHE_DIR": "/tmp/sleep-pretrained-uv-cache",
    "UV_PYTHON_INSTALL_DIR": "/tmp/sleep-pretrained-python",
    "PYTHONPATH": str(PROJECT_ROOT),
    "MPLBACKEND": "Agg",  # Backend только для дочернего CLI, не Kaggle-ядра.
}
UV_PROJECT = PROJECT_ROOT / "environments/pretrained"
print("Ядро notebook:", sys.version)
subprocess.run(["nvidia-smi"], check=True)
run_command(
    [
        UV,
        "sync",
        "--project",
        str(UV_PROJECT),
        "--locked",
        "--no-dev",
        "--python",
        "3.12.12",
    ],
    cwd=PROJECT_ROOT,
    env=ENV,
)
BASE = [
    UV,
    "run",
    "--project",
    str(UV_PROJECT),
    "--locked",
    "--no-dev",
    "python",
    "-m",
    "src.integration_run",
]
ENV.update(
    {
        k: str(THREADS)
        for k in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS")
    }
)
run_command(
    [
        UV,
        "run",
        "--project",
        str(UV_PROJECT),
        "--locked",
        "--no-dev",
        "python",
        "-c",
        "from src.pretrained import hardware; import json; print(json.dumps(hardware("
        + repr(DEVICE)
        + "),indent=2))",
    ],
    cwd=PROJECT_ROOT,
    env=ENV,
)

In [ ]:
import hashlib
from zipfile import ZipFile

import pandas as pd
from IPython.display import display


def unpack(path):
    path = Path(path)
    if path.is_dir():
        return path
    key = hashlib.sha256(
        (str(path) + str(path.stat().st_size) + str(path.stat().st_mtime_ns)).encode()
    ).hexdigest()[:16]
    target = Path("/tmp/sleep-edf-usleep-integration-inputs") / key
    if (target / ".unpacked").exists():
        return target
    target.mkdir(parents=True, exist_ok=True)
    with ZipFile(path) as z:
        for info in z.infolist():
            dest = (target / info.filename).resolve()
            if (
                not dest.is_relative_to(target.resolve())
                or (info.external_attr >> 16) & 0o170000 == 0o120000
            ):
                raise ValueError("Небезопасный путь в ZIP")
            z.extract(info, target)
    (target / ".unpacked").touch()
    return target


roots = [
    p.parent
    for p in Path("/kaggle/input").rglob("MANIFEST.json")
    if (p.parent / "labels.csv").is_file() and (p.parent / "epochs/val").is_dir()
]
if len(roots) != 1:
    raise RuntimeError(f"Нужен один исходный full_psg_v1; найдено: {roots}")
DATA_ROOT = roots[0]
if INPUT_ROOT is None:
    found = [
        p.parent
        for p in Path("/kaggle/input").rglob("bundle.json")
        if json.loads(p.read_text()).get("kind") == "usleep_integration_inputs_v1"
    ]
    if not found:
        for archive in Path("/kaggle/input").rglob("*.zip"):
            with ZipFile(archive) as z:
                if (
                    "bundle.json" in z.namelist()
                    and json.loads(z.read("bundle.json")).get("kind")
                    == "usleep_integration_inputs_v1"
                ):
                    found.append(unpack(archive))
    if len(found) != 1:
        raise RuntimeError(
            "Подключите один новый sleep-edf-usleep-integration-inputs.zip или укажите INPUT_ROOT явно"
        )
    INPUT_ROOT = found[0]
INPUT_ROOT = unpack(INPUT_ROOT)
for previous_input, phase in ((RESUME_PHASE1, "phase1"), (RESUME_PHASE2, "phase2")):
    if previous_input is None:
        continue
    previous = unpack(previous_input)
    for path in previous.rglob("*"):
        if (
            not path.is_file()
            or path.name == ".unpacked"
            or (path.name.endswith(".lock") and path.name != "uv.lock")
        ):
            continue
        target = OUTPUT_ROOT / phase / path.relative_to(previous)
        if target.exists():
            if (
                hashlib.sha256(target.read_bytes()).digest()
                != hashlib.sha256(path.read_bytes()).digest()
            ):
                raise ValueError(f"Resume conflict; preserved: {target}")
        else:
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(path, target)
COMMON = [
    "--data-root",
    str(DATA_ROOT),
    "--input-root",
    str(INPUT_ROOT),
    "--output-root",
    str(OUTPUT_ROOT),
    "--device",
    DEVICE,
    "--threads",
    str(THREADS),
]
print("DATA_ROOT:", DATA_ROOT, "\nINPUT_ROOT:", INPUT_ROOT)
print(
    "Phase1 не запускает модели. Phase2: один frozen проход U-Sleep по train, затем только Gate4/TCN4."
)

## Phase 1 — Quick ensemble ablations

160=A,161=B,162=C,163=D,164=D с поправками0.125. Проекция на вероятностный симплекс, диагностика поправок и сравнения с113/156/157.


In [ ]:
from datetime import UTC, datetime

from IPython.display import FileLink


def show_report(phase):
    path = (
        OUTPUT_ROOT
        / phase
        / ("phase1_report.json" if phase == "phase1" else "phase2_report.json")
    )
    if not path.exists():
        print(
            "Итоговый отчёт пока не готов; завершённые файлы и checkpoints сохранены."
        )
        return
    report = json.loads(path.read_text())
    if not report.get("complete", False):
        print("Неполный прогон:", report.get("error", report))
        return
    rows = report["results"] if phase == "phase1" else [report]
    table = []
    for row in rows:
        item = {k: v for k, v in row.items() if k not in ("f1", "components")}
        item.update(
            {"F1_" + s: v for s, v in zip(("W", "N1", "N2", "N3", "REM"), row["f1"])}
        )
        table.append(item)
    display(pd.DataFrame(table))


def export_phase(phase, cli_phase):
    stamp = datetime.now(UTC).strftime("%Y%m%dT%H%M%S%fZ")
    archive = Path("/kaggle/working") / f"sleep-edf-integration-{phase}-{stamp}.zip"
    run_command(
        [
            *BASE,
            "--output-root",
            str(OUTPUT_ROOT),
            "--phase",
            cli_phase,
            "--export",
            str(archive),
        ],
        cwd=PROJECT_ROOT,
        env=ENV,
    )
    display(
        FileLink(
            str(archive.relative_to(Path.cwd())),
            result_html_prefix=f"Скачать {phase}: ",
        )
    )
    return archive


run_command([*BASE, *COMMON, "--phase", "quick"], cwd=PROJECT_ROOT, env=ENV)
show_report("phase1")
PHASE1_ZIP = export_phase("phase1", "quick")
print("Phase1 полностью сохранена. Следующая ячейка обучает только новые heads.")

## Phase 2 — Train Gate4 + TCN4

Один frozen U-Sleep train inference кэшируется по записям. Старые nested OOF,106 features и Tiny128+AnySleep6 embeddings берутся из Input. По3 subject folds и refit для каждого head.
Если остановили обучение, следующую ячейку экспорта можно выполнить отдельно. Resume переносит Phase2 ZIP вместе с best/last и train probabilities; Phase1 ZIP подключается отдельно.


In [ ]:
if RUN_PHASE2:
    if not ALLOW_TRAINING:
        raise RuntimeError("Для запрошенного Phase2 требуется ALLOW_TRAINING=True")
    try:
        run_command(
            [*BASE, *COMMON, "--phase", "train", "--allow-training"],
            cwd=PROJECT_ROOT,
            env=ENV,
        )
    except subprocess.CalledProcessError:
        print(
            "Phase2 завершилась с ошибкой. Phase1 ZIP уже готов; следующая ячейка сохранит частичные checkpoints."
        )
    show_report("phase2")

In [ ]:
if (OUTPUT_ROOT / "phase2").is_dir():
    PHASE2_ZIP = export_phase("phase2", "train")
else:
    print("Phase2 ещё не запускалась. Phase1 ZIP уже сохранён отдельно.")